在 PyTorch 的优化器（torch.optim.Optimizer）中，param_groups 是一个列表（list），其中每个元素是一个字典（dict）。它是优化器管理模型参数的核心数据结构。
- 它的主要作用是允许用户对模型的不同部分设置不同的超参数（如学习率 lr、动量 momentum、权重衰减 weight_decay 等）。

## 结构解析

optimizer.param_groups 的结构如下：

In [ ]:
[
    {
        'params': [tensor1, tensor2, ...],  # 该组包含的参数张量列表
        'lr': 0.01,                         # 该组的学习率
        'momentum': 0.9,                    # 该组的动量
        'weight_decay': 0.0001,             # 该组的权重衰减
        'dampening': 0.0,                   # 该组的阻尼系数
        'nesterov': False,                  # 是否使用 Nesterov 动量
        ...                                 # 其他优化器特定的参数
    },
    {
        'params': [tensor3, tensor4, ...],  # 第二组参数
        'lr': 0.001,                        # 不同的学习率
        ...
    },
    ...
]

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 定义一个简单的模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)  # 包含 weight (5x10) 和 bias (5)
        self.fc2 = nn.Linear(5, 2)   # 包含 weight (2x5) 和 bias (2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleNet()

# 2. 查看模型原本的所有参数
print("--- 模型原始参数 ---")
for name, param in model.named_parameters():
    print(f"{name}: shape={param.shape}, id={id(param)}")
# 输出示例:
# fc1.weight: shape=torch.Size([5, 10])
# fc1.bias: shape=torch.Size([5])
# fc2.weight: shape=torch.Size([2, 5])
# fc2.bias: shape=torch.Size([2])

# 3. 构建包含 'params' 列表的优化器
# 这里我们手动将参数分组
optimizer = optim.SGD([
    # 【第一组】: 只包含 fc1 层的参数
    {
        'params': list(model.fc1.parameters()),  # <--- 关键点：这里是一个张量列表 [fc1.weight, fc1.bias]
        'lr': 0.01                               # fc1 使用小学习率
    },
    # 【第二组】: 只包含 fc2 层的参数
    {
        'params': list(model.fc2.parameters()),  # <--- 关键点：这里是一个张量列表 [fc2.weight, fc2.bias]
        'lr': 0.1                                # fc2 使用大学习率
    }
], momentum=0.9) # momentum 对两组都生效，除非在某组中单独覆盖

# 4. 深入检查 optimizer.param_groups 中的 'params'
print("\n--- 优化器参数组结构 ---")
for i, group in enumerate(optimizer.param_groups):
    print(f"\n第 {i} 组配置:")
    print(f"  学习率 (lr): {group['lr']}")
    print(f"  动量 (momentum): {group['momentum']}")
    
    # 这里是重点：查看 'params' 列表里到底存了什么
    param_list = group['params']
    print(f"  该组包含的参数数量: {len(param_list)}")
    
    for j, p in enumerate(param_list):
        # p 就是一个 torch.Tensor
        print(f"    - 参数 {j}: 形状 {p.shape}, 内存地址 {id(p)}, 是否需要梯度 {p.requires_grad}")
        
        # 验证：这个张量确实对应模型里的某一层
        # (在实际代码中通常不需要这样反向查找，这里仅为了演示)

# 5. 模拟一次训练步骤，观察 'params' 如何被使用
print("\n--- 模拟一次更新 ---")
dummy_input = torch.randn(1, 10)
target = torch.tensor([[1.0, 0.0]])

# 前向传播
output = model(dummy_input)
loss = nn.MSELoss()(output, target)

# 反向传播 (计算梯度，存入每个 tensor 的 .grad 属性)
loss.backward()

# 打印更新前的 fc1.weight (第一组第一个参数)
w1_before = optimizer.param_groups[0]['params'][0].data.clone()
w2_before = optimizer.param_groups[1]['params'][0].data.clone()

# 执行优化器步长 (优化器会遍历 param_groups -> params -> 更新每个 tensor)
optimizer.step()

# 打印更新后的值
w1_after = optimizer.param_groups[0]['params'][0].data
w2_after = optimizer.param_groups[1]['params'][0].data

# 计算变化量
diff1 = (w1_before - w1_after).abs().mean().item()
diff2 = (w2_before - w2_after).abs().mean().item()

print(f"第一组 (lr=0.01) 参数平均变化量: {diff1:.6f}")
print(f"第二组 (lr=0.1)  参数平均变化量: {diff2:.6f}")
print("可以看到，尽管梯度相同，但由于 'params' 分属不同组且 lr 不同，更新幅度也不同。")

--- 模型原始参数 ---
fc1.weight: shape=torch.Size([5, 10]), id=12934928864
fc1.bias: shape=torch.Size([5]), id=12934928944
fc2.weight: shape=torch.Size([2, 5]), id=12193498672
fc2.bias: shape=torch.Size([2]), id=12934928624

--- 优化器参数组结构 ---

第 0 组配置:
  学习率 (lr): 0.01
  动量 (momentum): 0.9
  该组包含的参数数量: 2
    - 参数 0: 形状 torch.Size([5, 10]), 内存地址 12934928864, 是否需要梯度 True
    - 参数 1: 形状 torch.Size([5]), 内存地址 12934928944, 是否需要梯度 True

第 1 组配置:
  学习率 (lr): 0.1
  动量 (momentum): 0.9
  该组包含的参数数量: 2
    - 参数 0: 形状 torch.Size([2, 5]), 内存地址 12193498672, 是否需要梯度 True
    - 参数 1: 形状 torch.Size([2]), 内存地址 12934928624, 是否需要梯度 True

--- 模拟一次更新 ---
第一组 (lr=0.01) 参数平均变化量: 0.000000
第二组 (lr=0.1)  参数平均变化量: 0.000000
可以看到，尽管梯度相同，但由于 'params' 分属不同组且 lr 不同，更新幅度也不同。


- <b>默认情况：</b>如果你初始化优化器时只传入了 model.parameters()，那么 param_groups 只有一个元素（即只有一组），所有参数共享相同的超参数。
- <b>多组情况：</b>你可以手动定义多个字典来创建多组，从而实现差异化配置。

## 为什么要使用 param_groups？（核心应用场景）
### 场景 A：分层学习率 (Layer-wise Learning Rates)

这是最常见的用法。在微调（Fine-tuning）预训练模型时，我们通常希望：
- 骨干网络（<b>Backbone</b>）：<b>使用较小的学习率</b>，以免破坏预训练的权重。
- 新添加的分类头（<b>Classifier Head</b>）：<b>使用较大的学习率</b>，以便快速收敛。

In [ ]:
import torch
import torch.optim as optim
import torchvision.models as models

model = models.resnet18(pretrained=True)

# 冻结骨干网络的梯度（可选，视策略而定）
# for param in model.features.parameters():
#     param.requires_grad = False

# 定义不同的参数组
optimizer = optim.SGD([
    {'params': model.layer1.parameters(), 'lr': 0.001}, # 浅层：小学习率
    {'params': model.layer2.parameters(), 'lr': 0.001}, # 深层：小学习率
    {'params': model.fc.parameters(), 'lr': 0.01}       # 分类头：大学习率 (默认其他参数如 momentum 会继承优化器默认值)
], lr=0.001, momentum=0.9) 
# 注意：这里的 lr=0.001 是默认值，如果某个组没指定 lr，就用这个。
# 上面显式指定的组会覆盖默认值。

# 验证
for i, group in enumerate(optimizer.param_groups):
    print(f"Group {i}: Learning Rate = {group['lr']}, Num Params = {len(group['params'])}")

###  不同的权重衰减（weight decay）

在某些高级架构（如 BatchNorm 层或 Bias 项）中，通常不建议应用权重衰减。你可以将参数分为两组：
- 需要权重衰减的参数（通常是权重 weight）。
- 不需要权重衰减的参数（通常是偏置 bias 和 BN 层的参数）。

In [ ]:
# 分离参数
decay_params = []
no_decay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    # 简单规则：如果是 bias 或 bn 层，不加 weight_decay
    if name.endswith('.bias') or 'bn' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = optim.SGD([
    {'params': decay_params, 'weight_decay': 1e-4},
    {'params': no_decay_params, 'weight_decay': 0.0} # 明确设为 0
], lr=0.01, momentum=0.9)

### 动态调整学习率

- 在训练循环中，你可以直接修改 param_groups 中的 lr 来实现自定义的学习率调度策略，而不必使用 torch.optim.lr_scheduler。

In [ ]:
# 在每个epoch结束时手动降低学习率

for param_group in optimizer.param_groups:
    param_group['lr'] *= 0.1 # 学习率每次乘以0.1

## 如何访问和操作

### 获取所有参数：

In [ ]:
all_params = []
for group in optimizer.param_groups:
    all_params.extend(group['params'])
# 或者直接使用
# all_params = list(optimizer.parameters()) 

### 查看特定组的配置：

In [ ]:
first_group = optimizer.param_groups[0]
print(first_group['lr'])
print(first_group['momentum'])

### 添加新的参数组（较少见，但在动态架构中可能用到）：

In [ ]:
new_layer = torch.nn.Linear(10, 10)
optimizer.add_param_group({'params': new_layer.parameters(), 'lr': 0.05})

| 特性 | 说明 |
| :--- | :--- |
| 数据类型 | `List[Dict[str, Any]]` |
| 核心键值 | `'params'` (必须), `'lr'`, `'momentum'`, `'weight_decay'` 等 |
| 主要用途 | 差异化配置。让模型的不同部分拥有不同的超参数。 |
| 典型应用 | 迁移学习（分层 LR）、正则化策略（区分 Bias/Weight）、动态调参。 |
| 注意事项 | 修改 `param_groups` 中的超参数会立即影响下一次 `optimizer.step()` 的行为。 |



# param.data
- param.data 是PyTorch中直接访问参数张量数据的属性，它允许绕过梯度跟踪系统，直接操作参数的底层数据。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

def basic_data_usage():
    """
    param.data 的基本作用
    """
    print("=== param.data 基本作用 ===\n")
    
    # 创建可训练参数
    param = nn.Parameter(torch.tensor([1.0, 2.0, 3.0]))
    
    print(f"参数对象: {param}")
    print(f"param.data: {param.data}")
    print(f"param.data 类型: {type(param.data)}")
    print(f"param.data requires_grad: {param.data.requires_grad}")
    print(f"原始参数 requires_grad: {param.requires_grad}")
    
    # 通过 param.data 修改数据
    param.data[0] = 100.0
    print(f"\n修改后参数: {param}")
    
    # 创建计算图
    loss = (param ** 2).sum()
    loss.backward()
    print(f"\n梯度: {param.grad}")
    
    # param.data 不会记录梯度
    with torch.no_grad():
        param.data[1] = 200.0
    print(f"\n使用 no_grad 修改后: {param}")
    print(f"梯度仍然存在: {param.grad}")

basic_data_usage()

=== param.data 基本作用 ===

参数对象: Parameter containing:
tensor([1., 2., 3.], requires_grad=True)
param.data: tensor([1., 2., 3.])
param.data 类型: <class 'torch.Tensor'>
param.data requires_grad: False
原始参数 requires_grad: True

修改后参数: Parameter containing:
tensor([100.,   2.,   3.], requires_grad=True)

梯度: tensor([200.,   4.,   6.])

使用 no_grad 修改后: Parameter containing:
tensor([100., 200.,   3.], requires_grad=True)
梯度仍然存在: tensor([200.,   4.,   6.])


# param.data vs param

In [3]:
def data_vs_param():
    """
    比较 param.data 和 param 的区别
    """
    print("=== param.data vs param ===\n")
    
    param = nn.Parameter(torch.tensor([1.0, 2.0, 3.0]))
    
    print("1. 属性对比:")
    print(f"  param.requires_grad: {param.requires_grad}")
    print(f"  param.data.requires_grad: {param.data.requires_grad}")
    print(f"  param.grad_fn: {param.grad_fn}")
    print(f"  param.data.grad_fn: {param.data.grad_fn}")
    
    print("\n2. 梯度跟踪对比:")
    # 计算图会跟踪 param
    y1 = param * 2
    print(f"  param * 2 有 grad_fn: {y1.grad_fn is not None}")
    
    # 但不会跟踪 param.data
    y2 = param.data * 2
    print(f"  param.data * 2 有 grad_fn: {y2.grad_fn is not None}")
    
    print("\n3. 内存地址对比:")
    print(f"  param 内存地址: {id(param)}")
    print(f"  param.data 内存地址: {id(param.data)}")
    print(f"  是否共享内存: {param.data.data_ptr() == param.data_ptr()}")
    
    print("\n4. 修改方式对比:")
    # 直接修改 param（会触发梯度跟踪）
    param[0] = 10.0  # 原地修改，影响梯度
    
    # 通过 data 修改（不影响梯度跟踪）
    param.data[1] = 20.0
    
    print(f"  修改后参数: {param}")

data_vs_param()

=== param.data vs param ===

1. 属性对比:
  param.requires_grad: True
  param.data.requires_grad: False
  param.grad_fn: None
  param.data.grad_fn: None

2. 梯度跟踪对比:
  param * 2 有 grad_fn: True
  param.data * 2 有 grad_fn: False

3. 内存地址对比:
  param 内存地址: 12986073872
  param.data 内存地址: 12986075232
  是否共享内存: True

4. 修改方式对比:


RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.